In [1]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# --------------------------------------------------
# Function 1 - Week 7
# --------------------------------------------------
# The Week 7 .npy files already contain the original
# Function 1 data plus Weeks 1-6 exactly once.
#
# Function 1 is treated as a true maximisation problem.
# I do NOT transform the outputs into "closeness to zero"
# because this changes the meaning of the objective.

In [2]:
X = np.load("function1/initial_inputs.npy")
Y = np.load("function1/initial_outputs.npy").reshape(-1)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

assert len(X) == len(Y)
assert X.shape[1] == 2

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("\nCurrent best observed input:", best_x)
print("Current best observed output:", best_y)

X shape: (16, 2)
Y shape: (16,)

Current best observed input: [0.73102363 0.73299988]
Current best observed output: 7.710875114502849e-16


In [3]:
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=np.ones(2) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-8,
        noise_level_bounds=(1e-12, 1e-2)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\nFitted kernel:")
print(gp.kernel_)


Fitted kernel:
1.04**2 * Matern(length_scale=[2, 0.0317], nu=2.5) + WhiteKernel(noise_level=1.79e-09)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [4]:
lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1 / lengthscales
sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:", lengthscales)
print("Normalised inverse-lengthscale sensitivity:", sensitivity)


ARD lengthscales: [2.         0.03167714]
Normalised inverse-lengthscale sensitivity: [0.01559162 0.98440838]


In [5]:
local_scale = np.clip(
    0.25 * lengthscales,
    0.02,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.05,
    0.20
)

print("\nLocal search widths:", local_scale)
print("Wider search widths:", wide_scale)


Local search widths: [0.1  0.02]
Wider search widths: [0.2  0.05]


In [6]:
rng = np.random.default_rng(42)

# Mostly refine around the best observation found automatically
local_candidates = best_x + rng.normal(
    0,
    local_scale,
    size=(40000, 2)
)

# Wider search around the same data-driven centre
wide_candidates = best_x + rng.normal(
    0,
    wide_scale,
    size=(20000, 2)
)

# Preserve some genuine global exploration
global_candidates = rng.uniform(
    0,
    1,
    size=(10000, 2)
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

candidates = np.clip(candidates, 0, 1)

print("Generated candidates:", len(candidates))

Generated candidates: 70000


In [7]:
tree = cKDTree(X)

distance, _ = tree.query(candidates, k=1)

candidates = candidates[distance > 0.01]

print("Candidates after filtering:", len(candidates))

Candidates after filtering: 68188


In [8]:
mu, sigma = gp.predict(
    candidates,
    return_std=True
)

In [9]:
def expected_improvement(mu, sigma, best_y, xi=0.0):

    improvement = mu - best_y - xi

    Z = np.zeros_like(mu)

    valid = sigma > 1e-12
    Z[valid] = improvement[valid] / sigma[valid]

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        + sigma[valid] * norm.pdf(Z[valid])
    )

    return EI

In [10]:
print("\nEI calibration:\n")

for xi in [0.0, 0.001, 0.005, 0.01]:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        f"xi={xi}",
        "\n candidate =", candidates[idx],
        "\n mean =", mu[idx],
        "\n std =", sigma[idx],
        "\n EI =", EI_test[idx],
        "\n"
    )


EI calibration:

xi=0.0 
 candidate = [0.91257214 0.5362032 ] 
 mean = -0.0001351604378752904 
 std = 0.0008647939681251291 
 EI = 0.00028162783119426197 

xi=0.001 
 candidate = [0.76381947 0.52130342] 
 mean = -0.00017092350895595156 
 std = 0.0008947973841359232 
 EI = 3.9999841991578545e-05 

xi=0.005 
 candidate = [0.076577   0.51317222] 
 mean = -0.0001823741573202265 
 std = 0.0008997145125998458 
 EI = 6.221765211879273e-13 

xi=0.01 
 candidate = [0.09342573 0.5116999 ] 
 mean = -0.00018315075038776758 
 std = 0.0008998088388238347 
 EI = 4.232425029358676e-34 



In [11]:
print("\nUCB calibration:\n")

for beta in [0.25, 0.5, 1.0, 1.5]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", mu[idx],
        "\n std =", sigma[idx],
        "\n UCB =", UCB[idx],
        "\n"
    )


UCB calibration:

beta=0.25 
 candidate = [0.03416915 0.74515982] 
 mean = 0.00011371734998269741 
 std = 0.00036720422729556295 
 UCB = 0.00020551840680658816 

beta=0.5 
 candidate = [0.95437043 0.54905435] 
 mean = -8.442494277567593e-05 
 std = 0.0007874896606927476 
 UCB = 0.0003093198875706979 

beta=1.0 
 candidate = [0.93112068 0.53398747] 
 mean = -0.00014210226177601315 
 std = 0.0008720633839621477 
 UCB = 0.0007299611221861346 

beta=1.5 
 candidate = [0.86559891 0.52469661] 
 mean = -0.00016493964692936074 
 std = 0.0008910980318436812 
 UCB = 0.001171707400836161 



In [12]:
# --------------------------------------------------
# Function 1 acquisition calibration
#
# Function 1 has an extremely small objective scale,
# so fixed xi values such as 0.001 or 0.01 are too large.
# Use xi values relative to the observed Y standard deviation.
# --------------------------------------------------

y_scale = np.std(Y)

print("Observed Y std:", y_scale)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\nScale-aware EI calibration:\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        f"xi={xi:.3e}",
        "\n candidate =", candidates[idx],
        "\n mean =", mu[idx],
        "\n std =", sigma[idx],
        "\n EI =", EI_test[idx],
        "\n"
    )

Observed Y std: 0.000872888785437539

Scale-aware EI calibration:

xi=0.000e+00 
 candidate = [0.91257214 0.5362032 ] 
 mean = -0.0001351604378752904 
 std = 0.0008647939681251291 
 EI = 0.00028162783119426197 

xi=8.729e-06 
 candidate = [0.91257214 0.5362032 ] 
 mean = -0.0001351604378752904 
 std = 0.0008647939681251291 
 EI = 0.0002778227909219336 

xi=4.364e-05 
 candidate = [0.91257214 0.5362032 ] 
 mean = -0.0001351604378752904 
 std = 0.0008647939681251291 
 EI = 0.00026294867056540024 

xi=8.729e-05 
 candidate = [0.93756411 0.53438749] 
 mean = -0.0001409001055397068 
 std = 0.0008708539219948756 
 EI = 0.00024518498746696916 



In [13]:
# Pure GP exploitation
mean_idx = np.argmax(mu)

print("Highest predicted mean:")
print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])

print("\nLow-exploration UCB:")

for beta in [0.1, 0.25, 0.5]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", mu[idx],
        "\n std =", sigma[idx],
        "\n"
    )

Highest predicted mean:
candidate = [0.50531106 0.74312993]
mean = 0.00013415988124542596
std = 0.0002260216931728155

Low-exploration UCB:
beta=0.1 
 candidate = [0.364627   0.74389532] 
 mean = 0.00013178227489862152 
 std = 0.0002690013466429881 

beta=0.25 
 candidate = [0.03416915 0.74515982] 
 mean = 0.00011371734998269741 
 std = 0.00036720422729556295 

beta=0.5 
 candidate = [0.95437043 0.54905435] 
 mean = -8.442494277567593e-05 
 std = 0.0007874896606927476 



In [14]:
# --------------------------------------------------
# Final Function 1 Week 7 acquisition
# --------------------------------------------------
#
# EI was found to favour high-uncertainty regions with negative predicted
# means. Since Function 1 already has a strong observed region and the fitted
# ARD kernel indicates that x2 is much more sensitive than x1, I use a
# low-exploration UCB acquisition for the final query.
#
# beta=0.1 still accounts for GP uncertainty, but places greater emphasis
# on the predicted mean rather than allowing uncertainty to dominate.

beta = 0.1

UCB = mu + beta * sigma
best_idx_week7 = np.argmax(UCB)

week7_candidate = candidates[best_idx_week7]

print("Week 7 Function 1 candidate:")
print(week7_candidate)

print("\nPredicted mean:", mu[best_idx_week7])
print("Predicted std:", sigma[best_idx_week7])
print("UCB:", UCB[best_idx_week7])

portal = "-".join(
    f"{x:.6f}" for x in week7_candidate
)

print("\nPortal format:")
print(portal)

Week 7 Function 1 candidate:
[0.364627   0.74389532]

Predicted mean: 0.00013178227489862152
Predicted std: 0.0002690013466429881
UCB: 0.00015868240956292033

Portal format:
0.364627-0.743895


In [15]:
# Week 7 reasoning:
# I calibrated EI using Function 1's actual output scale and compared it
# against several UCB exploration weights. EI repeatedly selected points
# with negative predicted means and very high uncertainty, showing that
# the recommendation was mainly exploration-driven.
#
# The fitted ARD Matérn kernel had lengthscales [2, 0.0317], meaning the GP
# currently models much faster variation in x2 than x1. The low-exploration
# UCB result kept x2 close to the strongest observed region while allowing
# a larger change in the less-sensitive x1 direction.
#
# I therefore used UCB with beta=0.1 for the final Week 7 query.